# 06 · Hybrid search and cross-encoder reranking

**Level:** intermediate to advanced · **Time:** 45–60 minutes · **Prerequisites:** notebooks 01–05.

Compare BM25 with dense retrieval, implement reciprocal rank fusion (RRF), and rerank a
candidate pool with a local cross-encoder. You will inspect ranking changes and verify source
coverage before generating an answer.

**Demo story:** “Exact identifiers and semantic meaning can require different retrieval methods.”
Live mode downloads embedding and cross-encoder models once and makes **one Claude call**.
Offline mode runs BM25 and lexical-vector fusion, labels that limitation, and skips neural
reranking and generation. Offline scores are not evidence of dense-retrieval quality.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## Step 1 · Build one shared corpus

Fair comparisons need identical chunks. BM25 rewards matching words and accounts for term
frequency and document length; dense retrieval compares learned vector representations.
They can rank the same documents differently, especially for product names, error codes,
abbreviations, and paraphrases. Neither method wins on every question.


In [ ]:
import time
import numpy as np

documents = load_documents()
chunks = chunk_documents(documents, chunk_size=75, overlap=15)
index = DenseIndex(chunks)
print("Vector retrieval backend:", index.backend)
question = "Team API gives HTTP 429. What limit applies and how should I retry?"
keyword_hits = bm25_search(question, chunks, k=6)
vector_hits = index.search(question, k=6)
print("BM25 ranking")
display(show_hits(keyword_hits))
print("Vector ranking")
display(show_hits(vector_hits))


## Step 2 · Fuse ranks instead of adding incompatible scores

BM25 and cosine similarity have different scales. Reciprocal rank fusion uses position:
each list contributes `1 / (rrf_k + rank)`, with rank starting at 1. A document present near
the top of several lists accumulates support. The constant dampens differences between ranks;
it is distinct from the number of final results, `k`.

RRF is a retrieval heuristic, not a probability model. Keep the original scores visible when
debugging, but do not interpret the sum as model confidence.


In [ ]:
def explain_rrf(*rankings, k=6, rrf_k=60):
    totals, records = {}, {}
    for ranking in rankings:
        seen = set()
        for rank, hit in enumerate(ranking, start=1):
            if hit["id"] in seen:
                continue
            seen.add(hit["id"])
            records[hit["id"]] = hit
            totals[hit["id"]] = totals.get(hit["id"], 0.0) + 1.0 / (rrf_k + rank)
    ranked_ids = sorted(totals, key=lambda record_id: (-totals[record_id], record_id))
    return [{**records[record_id], "score": totals[record_id]} for record_id in ranked_ids[:k]]

fused_hits = explain_rrf(keyword_hits, vector_hits, k=6)
display(show_hits(fused_hits))
print("A rank-1 contribution:", round(1 / 61, 6))
print("A document ranked first in both lists:", round(2 / 61, 6))


## Step 3 · Compare rank positions side by side

A score-only table can hide which source moved. This table keeps chunk IDs and missing
ranks visible. Inspect whether `api_limits` is represented before calling any generator.
A reranker cannot recover a source that never enters its candidate pool.


In [ ]:
def rank_table(**rankings):
    rows = {}
    for method, hits in rankings.items():
        for rank, hit in enumerate(hits, start=1):
            row = rows.setdefault(hit["id"], {
                "id": hit["id"], "source": hit["metadata"]["source_id"]
            })
            row[method] = rank
    return pd.DataFrame(rows.values()).set_index("id")

display(rank_table(BM25=keyword_hits, vector=vector_hits, RRF=fused_hits))
print("Fused source coverage:", sorted({h["metadata"]["source_id"] for h in fused_hits}))
assert all(h["id"] in {c.id for c in chunks} for h in fused_hits)


## Step 4 · Rerank query–document pairs with a cross-encoder

A bi-encoder embeds documents independently and can search precomputed vectors. A
cross-encoder reads the query and each candidate together, which costs more work per query.
Use it on a small candidate set instead of every document in a large collection.

The next cell uses the real `cross-encoder/ms-marco-MiniLM-L-6-v2` model locally. Its raw
scores rank pairs; they are not calibrated answer-correctness probabilities. The first live
run downloads model weights. Offline mode **does not simulate** these scores.


In [ ]:
reranked_hits = None
rerank_seconds = None
if offline_mode():
    print("OFFLINE: cross-encoder download and neural reranking skipped.")
else:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
    start = time.perf_counter()
    pair_scores = np.asarray(reranker.predict(
        [(question, hit["text"]) for hit in fused_hits], show_progress_bar=False
    )).reshape(-1)
    order = np.argsort(-pair_scores, kind="stable")
    reranked_hits = [
        {**fused_hits[i], "rrf_score": fused_hits[i]["score"],
         "score": float(pair_scores[i])} for i in order
    ]
    rerank_seconds = time.perf_counter() - start
    display(show_hits(reranked_hits))
    print("Reranking seconds (excludes model load):", round(rerank_seconds, 3))


## Step 5 · Compare the full retrieval path

Inspect top-three coverage, source duplication, and the first source supporting the exact
request limit. Reranking can help, do nothing, or make a ranking worse on this corpus.
One demonstration question does not establish overall superiority.


In [ ]:
rankings = {"BM25": keyword_hits, "vector": vector_hits, "RRF": fused_hits}
if reranked_hits is not None:
    rankings["cross_encoder"] = reranked_hits
display(rank_table(**rankings))
coverage_rows = [
    {"method": method,
     "api_limits_in_top3": any(h["metadata"]["source_id"] == "api_limits" for h in hits[:3]),
     "unique_sources_top3": len({h["metadata"]["source_id"] for h in hits[:3]})}
    for method, hits in rankings.items()
]
display(pd.DataFrame(coverage_rows))


## Step 6 · Generate only after choosing the evidence

Use the top three reranked candidates when a cross-encoder actually ran. Offline mode
explicitly uses the fusion candidate order and skips Claude. A larger candidate pool can
improve recall, while a smaller final context can reduce irrelevant evidence and token usage.

**API cost:** one Claude request in live mode, subject to the helper's local response cache.
Expected facts: Team **600/minute**, **HTTP 429**, and **Retry-After** handling.


In [ ]:
selected_hits = (reranked_hits if reranked_hits is not None else fused_hits)[:3]
print("Selection:", "cross-encoder" if reranked_hits is not None else "RRF only (offline)")
print(context_text(selected_hits))
answer = rag_answer(question, selected_hits, max_tokens=350)
display(Markdown(answer["text"]))
print({key: answer[key] for key in ("input_tokens", "output_tokens", "cached", "offline")})


## Step 7 · Try exact words and paraphrases

The questions below keep the corpus fixed and change only the wording. Record which
method puts the expected source highest. This is a small labelled retrieval probe, separate
from checking the correctness of generated answers; it makes no Claude calls.


In [ ]:
probes = [
    ("HTTP 429 Retry-After Team requests per minute", "api_limits"),
    ("Can I get my money back for a subscription I never used?", "refunds"),
    ("Can I save unused holiday entitlement for next year?", "benefits_leave"),
]
probe_rows = []
for query, expected_source in probes:
    bm = bm25_search(query, chunks, k=6)
    dense = index.search(query, k=6)
    fused = explain_rrf(bm, dense, k=6)
    for method, hits in (("BM25", bm), ("vector", dense), ("RRF", fused)):
        first_rank = next((rank for rank, hit in enumerate(hits, 1)
                           if hit["metadata"]["source_id"] == expected_source), None)
        probe_rows.append({"query": query, "method": method,
                           "expected_source": expected_source, "first_relevant_rank": first_rank})
display(pd.DataFrame(probe_rows))


## Step 8 · Candidate count and filter consistency

Apply the same eligible-document filter to **both** retrieval branches before fusion.
Otherwise an unfiltered branch can reintroduce excluded records. This pattern also applies
to source selection, department scoping, and document-version filters.

The following cell compares candidate counts without reranking or generation.


In [ ]:
api_doc = next(d for d in documents if d.id == "api_limits")
filters = {"department": api_doc.metadata["department"]}
pool_rows = []
for candidate_k in (2, 4, 8):
    bm = bm25_search(question, chunks, k=candidate_k, filters=filters)
    dense = index.search(question, k=candidate_k, filters=filters)
    fused = explain_rrf(bm, dense, k=candidate_k)
    assert all(h["metadata"]["department"] == filters["department"] for h in fused)
    pool_rows.append({"candidate_k": candidate_k, "actual_pool_size": len(fused),
                      "source_ids": sorted({h["metadata"]["source_id"] for h in fused})})
display(pd.DataFrame(pool_rows))


## Your turn

1. Change RRF's damping constant from 60 to 10 and compare the top three IDs.
2. Find a question where BM25 or dense retrieval does better on the expected source.
3. Explain why increasing final context size is different from increasing candidate-pool size.

**Presenter checkpoint:** show retrieval lists → fused candidates → reranked candidates →
selected context → answer. Do not describe offline lexical fusion as dense hybrid retrieval.


In [ ]:
# Sample solution: inspect scores and order rather than asserting an improvement.
comparison = {}
for damping in (10, 60):
    comparison[f"RRF_constant_{damping}"] = explain_rrf(
        keyword_hits, vector_hits, k=3, rrf_k=damping
    )
display(rank_table(**comparison))
print("A larger candidate pool gives the reranker more options.")
print("A larger final context sends more text to the generator and may add distraction.")


## What you can now demonstrate

Complementary retrievers, rank-based fusion, local pairwise reranking, and source-aware
comparisons. Tune with representative labelled questions; measure recall, answer support,
latency, and cost together instead of selecting a pipeline from one impressive example.

**Next:** notebook 07 broadens queries and balances relevance with coverage.

Primary references: [Sentence Transformers retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html),
[cross-encoder model card](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2),
[RRF paper](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf),
[rank-bm25 source](https://github.com/dorianbrown/rank_bm25).
